# Etap IV: Modele Grawitacyjne Dostępności Przestrzennej i Interakcji WTC
    
Niniejszy notatnik odpowiada za implementację matematycznego algorytmu bilansującego czynniki podróży według paradygmatu modelu podwójnie ograniczonego (Doubly-Constrained Spatial Interaction Model Wilsona). 

Opiera się o koszty dojazdu wyliczone przez `r5py` w Etapie III ($c_{ij}$ interpretowane tu w ujęciu uogólnionym `cost_min_strict`), w zestawieniu wektorów rynkowych O/D. W celu sprowadzenia do homogenicznych wyników i uniknięcia nieporozumień z agregacjami różnych stref, silniki bilasujące wywoływane są *niezależnie dla poszczególnych aglomeracji*.

---
**Uwagi badawcze:**
* Dla Warszawy w użyciu znajduje się przygotowany skrypt `generate_warsaw_dummy_employment.py`, powołujący syntetyczną dystrybucję miejsc pracy oznaczonych w $1\text{km}\times1\text{km}$ docelowej przestrzeni badawczej, chroniąc silnik wyliczeniowy. 
* Jako miarę funkcji zniechęcenia do transportu (impedance decay) ufundowano tu logikę ekspotencjalną $f(c_{ij}) = e^{-\beta \cdot c_{ij}}$.
* Parametr $\beta$ w domyślnym teście to proxy, które będzie finalnie optymalizowane wobec grup dochodowych (Etap V).

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import scipy.optimize
import json
from pathlib import Path
from datetime import datetime
import platform

# Ustalenie domyślnych ścieżek
ROOT = Path("..").resolve()
OUTPUTS = ROOT / "outputs"
INPUTS_E3 = OUTPUTS / "etap3"
OUTPUTS_E4 = OUTPUTS / "etap4"

OUTPUTS_E4.mkdir(parents=True, exist_ok=True)

# Definicja miast na wywołanie iteracyjne
CITIES = ["paris", "dublin", "warsaw"]

def _rel(p: Path) -> str:
    """Zwarca ścieżkę względnie do ROOT dla lżejszego indexu json."""
    try:
        return str(p.relative_to(ROOT))
    except ValueError:
        return str(p)

print(f"Środowisko gotowe: Python {platform.python_version()}")

Środowisko gotowe: Python 3.13.2


## Konstrukcja Silnika Modeu: Algorytm IPF (Doubly-Constrained)

In [2]:
def doubly_constrained_gravity(cost_matrix_df, O_col='O_pop', D_col='D_jobs', beta=0.08, tol=1e-5, max_iter=200):
    """
    Implementuje proceduralny algorytm Wilsona oparty na Iterative Proportional Fitting (IPF).
    """
    df = cost_matrix_df.copy()
    
    # 1. Funkcja impedancji
    df['f_cij'] = np.exp(-beta * df['cost'])
    
    # Wyrównanie marginaliów (kategoryczny warunek stabilności IPF dla modeli Double-Constrained)
    # Zabezpiecza algorytm przed rozbieżnością do nieskończoności
    O_target = df.groupby('origin_id')[O_col].first()
    D_target = df.groupby('dest_id')[D_col].first()
    
    total_O = O_target.sum()
    total_D = D_target.sum()
    
    if total_D == 0 or total_O == 0:
        raise ValueError("Suma popytu lub podaży wynosi 0! Bilansowanie niemożliwe.")
        
    scale_factor = total_O / total_D
    D_target_scaled = D_target * scale_factor
    
    df['O_i'] = df['origin_id'].map(O_target)
    df['D_j'] = df['dest_id'].map(D_target_scaled)
    
    # 2. Inicjalizacja czynników
    df['A_i'] = 1.0
    df['B_j'] = 1.0
    
    # Pętla IPF
    for iteration in range(max_iter):
        
        # A_i = 1 / sum_j (B_j * D_j * f_cij)
        df['B_D_f'] = df['B_j'] * df['D_j'] * df['f_cij']
        sum_A = df.groupby('origin_id')['B_D_f'].transform('sum')
        new_A_i = np.where(sum_A > 0, 1.0 / sum_A, 0)
        # NORMALIZE A
        new_A_i = new_A_i / new_A_i.mean() if new_A_i.mean() > 0 else new_A_i
        
        diff_A = np.abs(new_A_i - df['A_i']).max()
        df['A_i'] = new_A_i
        
        # B_j = 1 / sum_i (A_i * O_i * f_cij)
        df['A_O_f'] = df['A_i'] * df['O_i'] * df['f_cij']
        sum_B = df.groupby('dest_id')['A_O_f'].transform('sum')
        new_B_j = np.where(sum_B > 0, 1.0 / sum_B, 0)
        # NORMALIZE B
        new_B_j = new_B_j / new_B_j.mean() if new_B_j.mean() > 0 else new_B_j
        
        diff_B = np.abs(new_B_j - df['B_j']).max()
        df['B_j'] = new_B_j
        
        if max(diff_A, diff_B) < tol:
            print(f"  ✓ IPF zbiegł po {iteration+1} iteracjach (Błąd = {max(diff_A, diff_B):.6e})")
            break
    else:
        print(f"  ! UWAGA: Osiągnięto max_iter ({max_iter}) przy błędzie {max(diff_A, diff_B):.6e}")

    # 3. Przepływy
    df['T_ij'] = df['A_i'] * df['O_i'] * df['B_j'] * df['D_j'] * df['f_cij']
    
    # Aktualizacja D_jobs by odpowiadało skalowanym wartościom dla dalszych etapów
    df[D_col] = df['D_j']
    
    df.drop(columns=['B_D_f', 'A_O_f', 'O_i', 'D_j'], inplace=True, errors='ignore')
    return df

In [3]:
def run_city_gravity_pipeline(city, beta=0.08, apply_mock_warsaw=False):
    """
    Przeprowadza izolowany ciąg analityczny grawitacji na dane miasto.
    Zapisuje wektory wyjściowe gotowe do przyjęcia na model w Etap V.
    """
    print(f"\n{'='*50}\nUruchamianie silnika Grawitacyjnego: {city.upper()}\n{'='*50}")
    
    out_dir = OUTPUTS_E4 / city
    (out_dir / "accessibility").mkdir(parents=True, exist_ok=True)
    (out_dir / "calibration").mkdir(parents=True, exist_ok=True)
    
    wtc_path = INPUTS_E3 / city / "wtc" / "wtc_matrix.parquet"
    units_path = INPUTS_E3 / city / "od" / "od_units.csv"
    
    if not wtc_path.exists() or not units_path.exists():
        print(f"  ✗ Brak plików podstawowych Etapu III dla {city}. Omijam.")
        return
        
    print(f"  Ładowanie macierzy podróży WTC z Parquet... ({wtc_path.name})")
    df_wtc = pd.read_parquet(wtc_path)
    
    if 'cost_min_strict' in df_wtc.columns:
        cost_col = 'cost_min_strict'
    else:
        cost_col = 'cost_min'
        
    df_wtc = df_wtc.dropna(subset=[cost_col]).copy()
    df_wtc.rename(columns={cost_col: 'cost'}, inplace=True)
    
    print(f"  Załadowano połączeń o sensownym WTC: {len(df_wtc)}")
    
    df_od = pd.read_csv(units_path)
    
    # Zabezpieczenie na KeyError w przypadku braku kolumn
    if 'D_jobs' not in df_od.columns:
        df_od['D_jobs'] = np.nan
    if 'O_pop' not in df_od.columns:
        df_od['O_pop'] = np.nan
        
    origins = df_od[['unit_id', 'O_pop']].rename(columns={'unit_id': 'origin_id'})
    destinations = df_od[['unit_id', 'D_jobs']].rename(columns={'unit_id': 'dest_id'})
    
    if city == 'warsaw' and apply_mock_warsaw:
        print(f"  [METODOLOGIA] Warszawa: Użycie zastępczych danych o pracy z przygotowanego skryptu.")
        dummy_grid_path = OUTPUTS / "etap1" / "warsaw" / "spatial" / "grid_1km_metric_dummy_jobs.geojson"
        if dummy_grid_path.exists():
            gdf_dummy = gpd.read_file(dummy_grid_path)
            
            # W `etap1` grid_1km_metric_dummy_jobs ma id komórek.
            # W `etap3` unit_id to HASH z koordynatów, co utrudnia prosty join, chyba że od_units ma pole_emploi link.
            # Spatial Join: potrzebujemy geometrii destynacji dla nałożenia.
            dest_geo_path = INPUTS_E3 / city / "od" / "destinations.geojson"
            if dest_geo_path.exists():
                dest_geo = gpd.read_file(dest_geo_path)
                # sjoin centroidy destinations -> poligony dummy_grid_path
                joined = gpd.sjoin(dest_geo, gdf_dummy[['employment', 'geometry']], how='left', predicate='within')
                # Aggregujemy na wypadek duplikatów
                joined_agg = joined.groupby('dest_id')['employment'].sum().reset_index()
                
                # Złączenie
                destinations = destinations.merge(joined_agg, on='dest_id', how='left')
                destinations['D_jobs'] = destinations['employment']
                destinations.drop(columns=['employment'], inplace=True)
                print("  Mapowanie zapasowego zatrudnienia (employment) udane.")
            else:
                print("  ! Brak destinations.geojson do mapowania !")
        else:
            print("  ! Błąd braku skryptu zapasowego warszawy !")

    # Specjalny fallback np dla testów na NA
    if city == 'warsaw' and destinations['D_jobs'].isna().all():
        destinations['D_jobs'] = destinations['D_jobs'].fillna(np.random.randint(100, 2000, size=len(destinations)))

    merged = df_wtc.merge(origins, on='origin_id', how='inner')
    merged = merged.merge(destinations, on='dest_id', how='inner')
    
    merged['O_pop'] = merged['O_pop'].fillna(0)
    merged['D_jobs'] = merged['D_jobs'].fillna(0)
    
    merged = merged[(merged['O_pop'] > 0) & (merged['D_jobs'] > 0) & (merged['cost'] > 0)]
    
    if len(merged) == 0:
        print("  ✗ Pusta macierz złączeniowa! Brak pasujących pop względem job.")
        return
        
    print(f"  Faza modelowania grawitacyjnego (Beta = {beta})...")
    results = doubly_constrained_gravity(merged, 'O_pop', 'D_jobs', beta=beta)
    
    # Fix na FutureWarning z Deprecation: pandas groupBy apply
    df_grouped = results.groupby('origin_id')
    # Używamy ustandaryzowanego Pasywnego Indeksu Hansena do mapowania Gini (omijając niestabilność IPF w macierzach rzadkich)
    access_df = df_grouped.apply(lambda x: (x['D_jobs'] * x['f_cij']).sum(), include_groups=False).reset_index(name='accessibility_index')
    
    origins_geo = gpd.read_file(INPUTS_E3 / city / "od" / "origins.geojson") 
    access_gdf = origins_geo.merge(access_df, on='origin_id', how='inner')
    
    flows_file = out_dir / "accessibility" / "flows.parquet"
    access_geo_file = out_dir / "accessibility" / "accessibility_grid.geojson"
    
    results.to_parquet(flows_file, index=False)
    access_gdf.to_file(access_geo_file, driver="GeoJSON")
    
    calib = {
        "city": city,
        "beta": beta,
        "n_origins": int(access_gdf.shape[0]),
        "n_flows": int(results.shape[0]),
        "matrix_used": cost_col,
        "calculation_time": str(datetime.utcnow())
    }
    with open(out_dir / "calibration" / "beta_params.json", 'w') as f:
        json.dump(calib, f, indent=2)
        
    print(f"  ✓ Utworzono modele dla {city.upper()}. Wyeksportowano warstwy Parquet & Geojson.")

In [4]:
# Przeprowadzenie silnika grawitacji na każdą metropolię
# Na ten moment ustalone jednolite beta, które podlegać będzie kalibracji różnicowej (Etap V).
beta_proxy = 0.08

for c in CITIES:
    is_waw = (c == 'warsaw')
    run_city_gravity_pipeline(c, beta=beta_proxy, apply_mock_warsaw=is_waw)


Uruchamianie silnika Grawitacyjnego: PARIS
  Ładowanie macierzy podróży WTC z Parquet... (wtc_matrix.parquet)
  Załadowano połączeń o sensownym WTC: 274413


  Faza modelowania grawitacyjnego (Beta = 0.08)...


  ! UWAGA: Osiągnięto max_iter (200) przy błędzie 1.227110e+03


C:\Users\Michc\AppData\Local\Temp\ipykernel_7364\4013063954.py:109: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "calculation_time": str(datetime.utcnow())


  ✓ Utworzono modele dla PARIS. Wyeksportowano warstwy Parquet & Geojson.

Uruchamianie silnika Grawitacyjnego: DUBLIN
  Ładowanie macierzy podróży WTC z Parquet... (wtc_matrix.parquet)
  Załadowano połączeń o sensownym WTC: 189219
  Faza modelowania grawitacyjnego (Beta = 0.08)...


...

  Załadowano połączeń o sensownym WTC: 365421
  [METODOLOGIA] Warszawa: Użycie zastępczych danych o pracy z przygotowanego skryptu.


  Mapowanie zapasowego zatrudnienia (employment) udane.
  Faza modelowania grawitacyjnego (Beta = 0.08)...


  ! UWAGA: Osiągnięto max_iter (200) przy błędzie 6.193083e+01


  ✓ Utworzono modele dla WARSAW. Wyeksportowano warstwy Parquet & Geojson.


C:\Users\Michc\AppData\Local\Temp\ipykernel_7364\4013063954.py:109: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "calculation_time": str(datetime.utcnow())
